# Por qué el segmento manual

El segmento `state='manual'` fue elegido como foco del proyecto porque concentra
la mayor parte del churn real y muestra una señal estable a lo largo del tiempo.
Este notebook reproduce el análisis que motivó esa elección.

In [1]:
import sys
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import date
from pathlib import Path

HERE = Path.cwd().resolve()
for _p in [HERE, *HERE.parents]:
    if (_p / 'eda' / '_common.py').exists():
        sys.path.insert(0, str(_p / 'eda'))
        break
import _common as C

FEB_CUTOFF = date(2017, 1, 31)
MAR_CUTOFF = date(2017, 2, 28)
STATE_ORDER = ['auto-renew', 'manual', 'cancel + payment', 'cancel', 'no txn']

In [ ]:
rows = []
for label, cutoff in [('February', FEB_CUTOFF), ('March', MAR_CUTOFF)]:
    C.set_cutoff(cutoff)
    if label == 'February':
        users = C.load_cohort().select('msno', 'is_churn')
    else:
        users = pl.read_parquet(C.PARQUET_DIR / 'train_v2.parquet', columns=['msno', 'is_churn'])
    txn   = C.clean_transactions(users)
    state = C.subscription_state(txn, users)
    df    = state.join(users.select('msno', 'is_churn'), on='msno').to_pandas()
    for s in STATE_ORDER:
        g = df[df['state'] == s]
        if len(g) == 0:
            continue
        rows.append({
            'month':      label,
            'state':      s,
            'users':      len(g),
            'churners':   int(g['is_churn'].sum()),
            'churn_rate': g['is_churn'].mean(),
        })

stats = pd.DataFrame(rows)
print(stats.to_string(index=False))


In [ ]:
COLORS = {
    'auto-renew':       '#4b8f8c',
    'manual':           '#d66b4e',
    'cancel + payment': '#e8b84b',
    'cancel':           '#888888',
    'no txn':           '#bbbbbb',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: churn rate by state ───────────────────────────────────────────
ax = axes[0]
for month, marker in [('February', 'o'), ('March', 's')]:
    sub = stats[stats['month'] == month].set_index('state').reindex(STATE_ORDER).dropna()
    ax.plot(sub.index, sub['churn_rate'] * 100,
            marker=marker, lw=1.8, label=month,
            color='#2f5d8a' if month == 'February' else '#d66b4e')
    for x, y in zip(sub.index, sub['churn_rate']):
        ax.annotate(f'{y:.0%}', (x, y * 100), textcoords='offset points',
                    xytext=(0, 7), ha='center', fontsize=8)
ax.set_title('Churn rate by state', fontweight='bold')
ax.set_ylabel('Churn rate (%)')
ax.set_ylim(0, 55)
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.3)
ax.legend()

# ── Panel 2: share of churners by state (February) ────────────────────────
ax = axes[1]
feb = stats[stats['month'] == 'February'].set_index('state').reindex(STATE_ORDER).dropna()
total_churners = feb['churners'].sum()
pct_churners   = feb['churners'] / total_churners * 100
bars = ax.bar(feb.index, pct_churners,
              color=[COLORS[s] for s in feb.index])
ax.bar_label(bars, labels=[f'{v:.1f}%' for v in pct_churners], padding=3, fontsize=9)
ax.set_title('% of total churners by state\n(February)', fontweight='bold')
ax.set_ylabel('Churners (%)')
ax.set_ylim(0, 70)
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.3)

# ── Panel 3: volume and churn rate — February stacked ─────────────────────
ax = axes[2]
nc = feb['users'] - feb['churners']
x  = np.arange(len(feb))
ax.bar(x, nc,              label='No churn', color='#4b8f8c', alpha=0.85)
ax.bar(x, feb['churners'], label='Churn',    color='#d66b4e', alpha=0.85, bottom=nc)
ax2 = ax.twinx()
ax2.plot(x, feb['churn_rate'] * 100, 'k--o', lw=1.5, markersize=5, label='Churn rate')
ax2.set_ylabel('Churn rate (%)', color='black')
ax2.set_ylim(0, 55)
ax.set_xticks(x, feb.index, rotation=25)
ax.set_title('Volume and churn rate by state\n(February)', fontweight='bold')
ax.set_ylabel('Users')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}K'))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper right')
ax.grid(axis='y', alpha=0.3)

fig.suptitle(
    'The manual segment concentrates churn with a stable signal across months',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()

FIGS = Path.cwd().resolve()
for _p in [FIGS, *FIGS.parents]:
    if (_p / 'eda' / '_common.py').exists():
        FIGS = _p / 'w1' / 'figures' / 'eda_manual'
        FIGS.mkdir(parents=True, exist_ok=True)
        break
fig.savefig(FIGS / 'why_manual_segment.png', dpi=150, bbox_inches='tight')
plt.show()

man = stats[(stats['month'] == 'February') & (stats['state'] == 'manual')].iloc[0]
print(f"\nManual (Feb): {man['users']:,} users · {man['churn_rate']:.1%} churn · "
      f"{man['churners']/total_churners:.1%} of all churners")
